In [1]:
import pandas as pd
import numpy as np
import torch

from transformers import BertTokenizer, BertForSequenceClassification
from transformers import TrainingArguments, Trainer

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report
from datasets import Dataset

In [2]:
csv_file = "/home/amyliu/Desktop/GR/Experiment1/environmental_claims_dataset_original.csv"

df = pd.read_csv(csv_file)

print(df.head())
print(df["label"].value_counts())

                                                text  label
0  The project will make a significant contributi...      1
1  So in -- first of all, the segments, we are --...      0
2  Building a smarter and stronger energy grid th...      0
3  Generally, first of all our Transmission depar...      0
4  But for the most part, our challenge is showin...      0
label
0    1982
1     665
Name: count, dtype: int64


In [3]:
class EnvDataset:
    def __init__(self, dataframe, tokenizer, max_length=128):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df.loc[idx, "text"]
        label = int(self.df.loc[idx, "label"])

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label)
        }

        return item

In [4]:
from sklearn.model_selection import StratifiedKFold

def evaluate_model_5fold(model_name, df, n_splits=5):
    print(f"\nEvaluating: {model_name}")

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    fold_results = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(df["text"], df["label"])):
        print(f"\n========== Fold {fold + 1}/{n_splits} ==========")

        train_df = df.iloc[train_idx]
        test_df = df.iloc[test_idx]

        tokenizer = BertTokenizer.from_pretrained(model_name)

        train_dataset = EnvDataset(train_df, tokenizer)
        test_dataset = EnvDataset(test_df, tokenizer)

        model = BertForSequenceClassification.from_pretrained(
            model_name,
            num_labels=2
        )

        training_args = TrainingArguments(
            output_dir=f"./results/{model_name.replace('/', '_')}_fold{fold + 1}",
            eval_strategy="no",
            save_strategy="no",
            logging_steps=50,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=16,
            num_train_epochs=3,
            learning_rate=2e-5,
            weight_decay=0.01,
            report_to="none"
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset
        )

        trainer.train()

        predictions = trainer.predict(test_dataset)

        preds = np.argmax(predictions.predictions, axis=1)
        labels = predictions.label_ids

        acc = accuracy_score(labels, preds)
        macro_f1 = f1_score(labels, preds, average="macro")
        micro_f1 = f1_score(labels, preds, average="micro")

        print(f"Fold {fold + 1} Accuracy: {acc:.4f}")
        print(f"Fold {fold + 1} Macro F1: {macro_f1:.4f}")
        print(f"Fold {fold + 1} Micro F1: {micro_f1:.4f}")

        fold_results.append({
            "model": model_name,
            "fold": fold + 1,
            "accuracy": acc,
            "macro_f1": macro_f1,
            "micro_f1": micro_f1
        })

        del model
        del trainer
        torch.cuda.empty_cache()

    fold_df = pd.DataFrame(fold_results)

    return {
        "model": model_name,
        "accuracy_mean": fold_df["accuracy"].mean(),
        "accuracy_std": fold_df["accuracy"].std(),
        "macro_f1_mean": fold_df["macro_f1"].mean(),
        "macro_f1_std": fold_df["macro_f1"].std(),
        "micro_f1_mean": fold_df["micro_f1"].mean(),
        "micro_f1_std": fold_df["micro_f1"].std()
    }

In [5]:
models = [
    "google-bert/bert-base-uncased",
    "allenai/scibert_scivocab_uncased",
    "nlpaueb/legal-bert-base-uncased",
    "ahmedrachid/FinancialBERT",
    "Lianglab/PharmBERT-uncased",
    "recobo/agriculture-bert-uncased",
    "recobo/chemical-bert-uncased"
]

In [6]:
results = []

for model_name in models:
    result = evaluate_model_5fold(model_name, df, n_splits=5)
    results.append(result)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("macro_f1_mean", ascending=False)

print(results_df)


Evaluating: google-bert/bert-base-uncased

========== Fold 1/5 ==========


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
50,0.461296
100,0.310148
150,0.240975
200,0.186494
250,0.172275
300,0.116409
350,0.086910


Fold 1 Accuracy: 0.8981
Fold 1 Macro F1: 0.8677
Fold 1 Micro F1: 0.8981

========== Fold 2/5 ==========


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

... (131 outputs left out)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: recobo/chemical-bert-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider t

Step,Training Loss
50,0.480065
100,0.368134
150,0.296116
200,0.226987
250,0.203759
300,0.148913
350,0.119489


Fold 5 Accuracy: 0.8488
Fold 5 Macro F1: 0.8039
Fold 5 Micro F1: 0.8488
                              model  accuracy_mean  accuracy_std  \
5   recobo/agriculture-bert-uncased       0.922176      0.007953   
1  allenai/scibert_scivocab_uncased       0.908203      0.017091   
0     google-bert/bert-base-uncased       0.904043      0.010289   
2   nlpaueb/legal-bert-base-uncased       0.902151      0.012301   
3         ahmedrachid/FinancialBERT       0.899130      0.009513   
4        Lianglab/PharmBERT-uncased       0.896485      0.013529   
6      recobo/chemical-bert-uncased       0.865125      0.015426   

   macro_f1_mean  macro_f1_std  micro_f1_mean  micro_f1_std  
5       0.898844      0.011027       0.922176      0.007953  
1       0.881339      0.021754       0.908203      0.017091  
0       0.878008      0.012450       0.904043      0.010289  
2       0.876135      0.014626       0.902151      0.012301  
3       0.868348      0.013063       0.899130      0.009513  
4       0.8

In [7]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "macro_f1_mean",
    ascending=False
)

print(results_df)

                              model  accuracy_mean  accuracy_std  \
5   recobo/agriculture-bert-uncased       0.922176      0.007953   
1  allenai/scibert_scivocab_uncased       0.908203      0.017091   
0     google-bert/bert-base-uncased       0.904043      0.010289   
2   nlpaueb/legal-bert-base-uncased       0.902151      0.012301   
3         ahmedrachid/FinancialBERT       0.899130      0.009513   
4        Lianglab/PharmBERT-uncased       0.896485      0.013529   
6      recobo/chemical-bert-uncased       0.865125      0.015426   

   macro_f1_mean  macro_f1_std  micro_f1_mean  micro_f1_std  
5       0.898844      0.011027       0.922176      0.007953  
1       0.881339      0.021754       0.908203      0.017091  
0       0.878008      0.012450       0.904043      0.010289  
2       0.876135      0.014626       0.902151      0.012301  
3       0.868348      0.013063       0.899130      0.009513  
4       0.867752      0.015798       0.896485      0.013529  
6       0.823770     